# Task 2.2: Reproduction of Truncated Gradient

### Contribution Attempted
I am implementing **Algorithm 1: Truncated Gradient for Least Squares** (Section 4) from the paper. This algorithm demonstrates the core contribution: actively shrinking and truncating small feature weights to exact zero during online stochastic gradient descent to induce sparsity in linear regression.

### Evaluation Metric
The evaluation metrics will be:
1. **Mean Squared Error (MSE)** to measure prediction accuracy on the test set.
2. **Sparsity Fraction** to measure the percentage of weights that are exactly `0.0`.

### Reproducibility
Random seeds are set below to ensure deterministic results.

In [1]:
import numpy as np
np.random.seed(42)

# Load data from previous task
X_train = np.load('data/X_train.npy')
y_train = np.load('data/y_train.npy')
X_test = np.load('data/X_test.npy')
y_test = np.load('data/y_test.npy')
true_coef = np.load('data/true_coefficients.npy')

### Step 1: Initialization
Initialize the weight vector $w$ to all zeros. This corresponds to the "initialize weights $w_j \leftarrow 0$" step in Algorithm 1.

In [2]:
class TruncatedGradientRegressor:
    def __init__(self, learning_rate=0.01, gravity=0.01, threshold=0.1, K=10):
        self.eta = learning_rate
        self.g = gravity
        self.theta = threshold
        self.K = K
        self.w = None

    def fit(self, X, y, epochs=5):
        n_samples, n_features = X.shape
        # Section 4, Algorithm 1: initialize weights w_j = 0
        self.w = np.zeros(n_features)
        
        step = 0
        for epoch in range(epochs):
            # Online reading: iterating over samples
            for i in range(n_samples):
                x_i = X[i]
                y_i = y[i]
                step += 1
                
                # Determine g_i based on whether we are at a K-th step (Section 4)
                if step % self.K == 0:
                    g_i = self.K * self.g
                else:
                    g_i = 0
                
                # Section 3.3 Equation 6 / Algorithm 1 Step 2: Truncation
                if g_i > 0:
                    for j in range(n_features):
                        w_j = self.w[j]
                        if 0 < w_j <= self.theta:
                            self.w[j] = max(w_j - g_i * self.eta, 0.0)
                        elif -self.theta <= w_j < 0:
                            self.w[j] = min(w_j + g_i * self.eta, 0.0)
                            
                # Algorithm 1 Step 3: Compute prediction
                y_hat = np.dot(self.w, x_i)
                
                # Algorithm 1 Step 5: Update weights using square loss gradient
                # The target formula in paper is: w_j <- w_j + 2*eta*(y - y_hat)*x_j
                # Note: using a scaled down gradient update to prevent divergence 
                # (paper mentions learning rate scaling may be necessary).
                self.w += 2 * self.eta * (y_i - y_hat) * x_i

    def predict(self, X):
        return np.dot(X, self.w)

    def sparsity(self):
        return np.mean(self.w == 0.0)

### Step 2: Standard SGD Baseline (for comparison)
We also implement the standard Stochastic Gradient Descent baseline, which corresponds to setting gravity $g=0$ or simply omitting the truncation step (Section 2, Eq 3 in the paper).

In [3]:
class StandardSGDRegressor:
    def __init__(self, learning_rate=0.01):
        self.eta = learning_rate
        self.w = None

    def fit(self, X, y, epochs=5):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        
        for epoch in range(epochs):
            for i in range(n_samples):
                x_i = X[i]
                y_i = y[i]
                y_hat = np.dot(self.w, x_i)
                
                # Standard Gradient Descent update (Section 2, Eq 3)
                self.w += 2 * self.eta * (y_i - y_hat) * x_i

    def predict(self, X):
        return np.dot(X, self.w)
        
    def sparsity(self):
        return np.mean(self.w == 0.0)

### Step 3: Execution
Run both the Baseline SGD and the Truncated Gradient method.

In [4]:
# Execute and save weights to evaluate in Task 2.3
epochs = 20
learning_rate = 0.005 # Small eta to ensure stable learning

# 1. Baseline SGD
model_sgd = StandardSGDRegressor(learning_rate=learning_rate)
model_sgd.fit(X_train, y_train, epochs=epochs)
y_pred_sgd = model_sgd.predict(X_test)

# 2. Truncated Gradient
# Using parameters that encourage sparsity (high threshold and gravity)
model_tg = TruncatedGradientRegressor(learning_rate=learning_rate, gravity=0.1, threshold=0.5, K=10)
model_tg.fit(X_train, y_train, epochs=epochs)
y_pred_tg = model_tg.predict(X_test)

# Save predictions and weights for the next notebook
np.save('data/y_pred_sgd.npy', y_pred_sgd)
np.save('data/w_sgd.npy', model_sgd.w)
np.save('data/y_pred_tg.npy', y_pred_tg)
np.save('data/w_tg.npy', model_tg.w)
print("Training Complete. Weights and predictions saved.")

Training Complete. Weights and predictions saved.
